In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
from unet_model import *
from unet_dataset import *
import matplotlib.gridspec as gridspec
from scipy.signal import welch
from scipy.stats import gmean

NB_ID = "40"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 1
P_WEIGHT = 0.11106880351917545

MODEL_PATH = MODELS_DIR / "lucidrf_unet_checkpoint.pth"

METADATA_PATH = get_unet_path(STAGE_SCALED, TEST, "mixed_metadata", extension="csv")
X_TEST_PATH = get_unet_path(STAGE_FINAL, TEST, "X_test")
Y_TEST_PATH = get_unet_path(STAGE_FINAL, TEST, "Y_test")

print(f"Running on Device: {DEVICE}")

In [ ]:
class PhaseAwareLoss(nn.Module):
    def __init__(self, phase_weight):
        """
        Combines standard MSE with a Cosine Similarity penalty to enforce 
        phase alignment for RF constellations (like QPSK).
        """
        super().__init__()
        self.mse = nn.MSELoss()
        self.phase_weight = phase_weight

    def forward(self, y_pred, y_true):
        # Standard Amplitude/Waveform Loss (Maintains volume stability)
        mse_loss = self.mse(y_pred, y_true)
        
        # Phase Alignment (Cosine Similarity)
        # y_pred and y_true are shape (Batch, 2, L) where dim=1 is the I/Q channel
        # Cosine similarity evaluates to 1 if perfectly aligned, -1 if opposite.
        cos_sim = F.cosine_similarity(y_pred, y_true, dim=1)
        
        # We want to minimize the difference from perfect alignment (1.0)
        # Taking the mean across the batch and sequence length
        phase_penalty = torch.mean(1.0 - cos_sim)
        
        # Combined Objective
        total_loss = mse_loss + (self.phase_weight * phase_penalty)
        
        return total_loss

In [ ]:
# Helper: Compute Power of a batch (B, 2, L) -> (B,)
def batch_power(tensor):
    # Sum of squares across channels and time, averaged by length
    # Note: Using mean preserves the scale relative to 1.0
    return torch.mean(tensor**2, dim=(1, 2))

# Helper: Compute SINR in dB for a batch
def batch_sinr_db(signal, noise):
    p_sig = batch_power(signal)
    p_noise = batch_power(noise)
    # Add epsilon for numerical stability
    return 10 * torch.log10(p_sig / (p_noise + 1e-10))

In [ ]:
print("\n[1/3] Loading Data...")
test_dataset = LucidRFDataset(X_TEST_PATH, Y_TEST_PATH)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
df = pd.read_csv(METADATA_PATH)
print(f"   Test Set Size: {len(test_dataset)} samples")

In [ ]:
print("\n[2/3] Loading Model...")
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = LucidRF_UNet(n_channels=2, n_classes=2).to(DEVICE)

# Load weights
if torch.cuda.is_available():
    state_dict = torch.load(MODEL_PATH)
else:
    state_dict = torch.load(MODEL_PATH, map_location=torch.device('cpu'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("   Weights loaded successfully.")

In [ ]:
print("\n[3/3] Running Evaluation Loop...")

mse_losses, sinr_inputs, sinr_outputs, flatness_scores = [], [], [], []
y_powers, input_evms, output_evms = [], [], []
psd_x_sum, psd_pred_sum, psd_y_sum = None, None, None
freqs = None # Will hold the frequency x-axis array
criterion = PhaseAwareLoss(phase_weight=P_WEIGHT).to(DEVICE)

with torch.no_grad():
    for x_batch, y_batch in tqdm(test_loader, desc="Evaluating"):
        x_batch = x_batch.to(DEVICE) # Noisy Input
        y_batch = y_batch.to(DEVICE) # Clean Target (Ground Truth)

        # Model Prediction
        y_pred = model(x_batch)

        # MSE Calculation
        loss_tensor = criterion(y_pred, y_batch)
        mse_losses.append(loss_tensor.item())

        # SINR Calculation
        # Input Noise = Input - Target
        noise_in = x_batch - y_batch
        
        # Output Noise (Residual) = Prediction - Target
        noise_out = y_pred - y_batch

        # Calculate dB metrics
        s_in_db = batch_sinr_db(y_batch, noise_in)
        s_out_db = batch_sinr_db(y_batch, noise_out)

        sinr_inputs.extend(s_in_db.cpu().numpy())
        sinr_outputs.extend(s_out_db.cpu().numpy())

        p = torch.mean(y_batch[0, 0]**2 + y_batch[0, 1]**2).item()
        y_powers.append(p)

        rms_ref = torch.sqrt(torch.mean(y_batch[0, 0]**2 + y_batch[0, 1]**2)).item()
        if rms_ref < 1e-12:
            input_evms.append(np.nan)
            output_evms.append(np.nan)
        else:
            rms_err_in = torch.sqrt(torch.mean(noise_in[0, 0]**2 + noise_in[0, 1]**2)).item()
            rms_err_out = torch.sqrt(torch.mean(noise_out[0, 0]**2 + noise_out[0, 1]**2)).item()
            input_evms.append((rms_err_in / rms_ref) * 100)
            output_evms.append((rms_err_out / rms_ref) * 100)

        y_np = y_batch.cpu().numpy()
        x_np = x_batch.cpu().numpy()
        pred_np = y_pred.cpu().numpy()

        for i in range(x_np.shape[0]):
            # Recombine I and Q channels into complex baseband
            c_x = x_np[i, 0] + 1j * x_np[i, 1]
            c_y = y_np[i, 0] + 1j * y_np[i, 1]
            c_pred = pred_np[i, 0] + 1j * pred_np[i, 1]
            
            # Isolate residual noise
            residual = c_x - c_pred
            
            # Calculate PSD and Flatness
            f_res, psd_res = welch(residual, fs=MIT_SAMPLE_RATE, return_onesided=False, nperseg=1024)
            
            # Add a tiny epsilon to prevent log/gmean errors if PSD hits pure 0
            psd_res = psd_res + 1e-12 
            
            score = gmean(psd_res) / np.mean(psd_res)
            flatness_scores.append(score)

            # --- Global PSD Accumulation ---
            f_x, psd_x = welch(c_x, fs=MIT_SAMPLE_RATE, return_onesided=False, nperseg=1024)
            _, psd_pred = welch(c_pred, fs=MIT_SAMPLE_RATE, return_onesided=False, nperseg=1024)
            _, psd_y = welch(c_y, fs=MIT_SAMPLE_RATE, return_onesided=False, nperseg=1024)

            # Shift zero frequency to center
            f_x = np.fft.fftshift(f_x)
            psd_x = np.fft.fftshift(psd_x)
            psd_pred = np.fft.fftshift(psd_pred)
            psd_y = np.fft.fftshift(psd_y)

            # Add to the running totals
            if psd_x_sum is None:
                psd_x_sum = psd_x
                psd_pred_sum = psd_pred
                psd_y_sum = psd_y
                freqs = f_x
            else:
                psd_x_sum += psd_x
                psd_pred_sum += psd_pred
                psd_y_sum += psd_y

In [ ]:
# Calculate the Final Averages for PSD
num_samples = len(test_dataset)
psd_x_avg = psd_x_sum / num_samples
psd_pred_avg = psd_pred_sum / num_samples
psd_y_avg = psd_y_sum / num_samples

global_flatness = np.mean(flatness_scores)


# Merge into DataFrame
df['Input_SINR_dB'] = np.array(sinr_inputs)
df['Output_SINR_dB'] = np.array(sinr_outputs)
df['Model_Gain_dB'] = df['Output_SINR_dB'] - df['Input_SINR_dB']
df['MSE_Loss'] = mse_losses[:len(df)]
df['Spectral_Flatness'] = flatness_scores
df['GT_Power'] = y_powers
df['Input_EVM_%'] = input_evms
df['Output_EVM_%'] = output_evms
df['EVM_Improvement_%'] = df['Input_EVM_%'] - df['Output_EVM_%']
POWER_THRESHOLD = np.percentile(df['GT_Power'], 75)
active_df = df[df['GT_Power'] >= POWER_THRESHOLD]

# Create SINR bins for stratified analysis
bins, labels = [-np.inf, -10, -5, 0, 5, 10, np.inf], ['<-10dB', '-10 to -5dB', '-5 to 0dB', '0 to 5dB', '5 to 10dB', '>10dB']
df['SINR_Bin'] = pd.cut(df['Input_SINR_dB'], bins=bins, labels=labels)


In [ ]:
print("\n" + "="*40 + "\n       EVALUATION REPORT\n" + "="*40)
print(f"Global MSE Loss   : {np.mean(mse_losses):.6f}")
print(f"Avg Input SINR    : {df['Input_SINR_dB'].mean():.2f} dB")
print(f"Avg SINR GAIN     : {df['Model_Gain_dB'].mean():.2f} dB")
print("="*40)
print("\n"+"="*40 + "\n       AVERAGE GAIN PER DIFFICULTY BIN\n" + "="*40)
print(df.groupby('SINR_Bin', observed=False)['Model_Gain_dB'].mean())
print("="*40)

print("\n" + "="*40)
print("             EVM PERFORMANCE")
print("="*40)
print(f"Active Packets Evaluated  : {len(active_df)}")
print(f"Average Input EVM (Noisy) : {active_df['Input_EVM_%'].mean():.2f}%")
print(f"Average Output EVM (Clean): {active_df['Output_EVM_%'].mean():.2f}%")
print(f"Average EVM Reduction     : {active_df['EVM_Improvement_%'].mean():.2f}%")
print("="*40)


print("\n"+"="*40 + "\n       RESIDUAL FORENSICS\n" + "="*40)
print("="*40)
print(f"Global Spectral Flatness  : {df['Spectral_Flatness'].mean():.4f}")
print(f"Residual Spectral Flatness: {global_flatness:.4f}")
print("-" * 40)
print("INTERPRETATION:")
print(" • Target: ~1.0 (Indicates perfectly flat, white noise).")
print(" • If the value is close to 1.0, the U-Net is accurately extracting only the Barrage jamming.")
print(" • If the value is significantly lower than 1.0, the residual contains structured peaks, meaning the model is accidentally deleting parts of the CommSignal2 payload.")

In [ ]:
# --- Plot A: Histogram of Gains ---
sns.histplot(df['Model_Gain_dB'], bins=50, kde=True, color='green', alpha=0.6)
plt.axvline(df['Model_Gain_dB'].mean(), color='red', linestyle='--', label=f"Mean: {df['Model_Gain_dB'].mean():.2f} dB")
plt.title('Distribution of SINR Improvement (Test Set)')
plt.xlabel('SINR Gain (dB)')
plt.legend()
save_plot("sinr_gain_histogram", machine_id="B", nb_id=NB_ID, fig_id="01")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Input_SINR_dB', y='Model_Gain_dB', alpha=0.6)
plt.axhline(0, color='red', linestyle='--', label='Zero Gain (Identity)')
plt.title('Impact of Input Signal Quality on Model Performance')
plt.xlabel('Input SINR (dB)')
plt.ylabel('SINR Gain (dB)')
plt.grid(True, alpha=0.3)
plt.legend()
save_plot("gain_vs_input_quality_scatter", machine_id="B", nb_id=NB_ID, fig_id="02")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.semilogy(freqs/1e6, psd_x_avg, label='Average Noisy Input (Barrage)', color='red', alpha=0.4)
plt.semilogy(freqs/1e6, psd_pred_avg, label='Average U-Net Prediction', color='blue', alpha=0.8, linewidth=1.5)
plt.semilogy(freqs/1e6, psd_y_avg, label='Average Ground Truth', color='green', alpha=0.8, linestyle='--')

plt.title('Spectral Denoising (Average PSD across Test Set)')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Average Power Spectral Density')
plt.legend()
plt.grid(True, alpha=0.3)
save_plot("global_psd_overlay", machine_id="B", nb_id=NB_ID, fig_id="03")
plt.show()

In [ ]:
# Find the index of the highest SINR gain
best_idx = df['Model_Gain_dB'].idxmax()
best_gain = df.loc[best_idx, 'Model_Gain_dB']
input_sinr = df.loc[best_idx, 'Input_SINR_dB']

print(f"Visualizing Best Success Story (Index {best_idx}):")
print(f"Input SINR: {input_sinr:.2f} dB -> Model Gain: +{best_gain:.2f} dB")

# Load the specific sample and run inference
x_best, y_best = test_dataset[best_idx]
x_batch = x_best.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    y_pred_best = model(x_batch).squeeze().cpu().numpy()

x_np = x_best.cpu().numpy()
y_np = y_best.cpu().numpy()

# Convert to complex arrays for spectrograms
c_x = x_np[0] + 1j * x_np[1]
c_pred = y_pred_best[0] + 1j * y_pred_best[1]
c_y = y_np[0] + 1j * y_np[1]

# Create a 3x3 plot figure
fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 3, hspace=0.3, wspace=0.2)

titles = [
    f"Noisy Input (Barrage)\nSINR: {input_sinr:.2f} dB", 
    f"U-Net Denoised\nGain: +{best_gain:.2f} dB", 
    "Clean Ground Truth\n(Target)"
]
signals = [c_x, c_pred, c_y]
raw_arrays = [x_np, y_pred_best, y_np]
colors = ['red', 'blue', 'green']

for i in range(3):
    # Time Domain (Real part - zooming in on first 1000 samples)
    ax_time = fig.add_subplot(gs[0, i])
    ax_time.plot(raw_arrays[i][0][:1000], color=colors[i], alpha=0.8, linewidth=1)
    ax_time.set_title(titles[i], fontsize=14, fontweight='bold')
    ax_time.set_ylabel('Amplitude (Real)')
    ax_time.set_xlabel('Sample Index')
    ax_time.grid(True, alpha=0.3)
    ax_time.set_ylim([-1.5, 1.5]) # Based on your global scaling
    
    # Row 2: Spectrogram (Frequency Domain over Time)
    ax_spec = fig.add_subplot(gs[1, i])
    Pxx, freqs, bins, im = ax_spec.specgram(signals[i], NFFT=1024, Fs=MIT_SAMPLE_RATE, noverlap=512, cmap='magma')
    ax_spec.set_ylabel('Frequency (Hz)')
    ax_spec.set_xlabel('Time (s)')
    
    # Constellation (I vs Q phase recovery)
    ax_const = fig.add_subplot(gs[2, i])
    ax_const.scatter(raw_arrays[i][0], raw_arrays[i][1], alpha=0.02, s=1, color=colors[i])
    ax_const.set_aspect('equal')
    ax_const.set_xlim([-1.5, 1.5])
    ax_const.set_ylim([-1.5, 1.5])
    ax_const.set_xlabel('In-Phase (I)')
    ax_const.set_ylabel('Quadrature (Q)')
    ax_const.grid(True, alpha=0.3)

plt.suptitle(f'Performance example sample: {best_idx}', fontsize=22, y=0.96)
# Save graphic using your existing pipeline utility
save_plot("best_success_story_grid", machine_id="B", nb_id=NB_ID, fig_id="04")
plt.show()

In [ ]:
# Calculate PSD
f_x, psd_x = welch(c_x, MIT_SAMPLE_RATE, return_onesided=False, nperseg=1024)
f_pred, psd_pred = welch(c_pred, MIT_SAMPLE_RATE, return_onesided=False, nperseg=1024)
f_y, psd_y = welch(c_y, MIT_SAMPLE_RATE, return_onesided=False, nperseg=1024)

# Shift zero frequency to center
f_x, psd_x = np.fft.fftshift(f_x), np.fft.fftshift(psd_x)
f_pred, psd_pred = np.fft.fftshift(f_pred), np.fft.fftshift(psd_pred)
f_y, psd_y = np.fft.fftshift(f_y), np.fft.fftshift(psd_y)

plt.figure(figsize=(12, 7))

# Plot the deafening noise (Red)
plt.semilogy(f_x/1e6, psd_x, label=f'Noisy Input (Barrage)', color='#e74c3c', alpha=0.5, linewidth=1)

# Plot the model's recovery (Blue)
plt.semilogy(f_pred/1e6, psd_pred, label=f'U-Net Prediction (+{best_gain:.2f} dB)', color='#3498db', alpha=0.9, linewidth=2)

# Plot the gold standard (Green)
plt.semilogy(f_y/1e6, psd_y, label='Clean Ground Truth', color='#2ecc71', alpha=0.9, linestyle='--', linewidth=2)

plt.title(f'Spectral Denoising (sample: {best_idx})', fontsize=18, fontweight='bold')
plt.xlabel('Frequency (MHz)', fontsize=14)
plt.ylabel('Power Spectral Density', fontsize=14)

# Add annotations to help laypeople understand
plt.annotate('Model suppressed this broadband noise', 
             xy=(-8, 1e-4), xytext=(-10, 1e-2),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
             fontsize=12, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

plt.legend(fontsize=12, loc='upper right')
plt.grid(True, alpha=0.3, which='both') # 'both' turns on minor grid lines for log scale
save_plot("best_sample_psd_overlay", machine_id="B", nb_id=NB_ID, fig_id="05")
plt.show()